# Topic Modeling

## Load the dataset

In [1]:
import pandas as pd

exploratory_df = pd.read_csv("dataset/processed/exploratory-packages-feature-engineering.csv")
confirmatory_df = pd.read_csv("dataset/processed/confirmatory-packages-feature-engineering.csv")
holdout_df = pd.read_csv("dataset/processed/holdout-packages-feature-engineering.csv")

exploratory_df.head()

,package_id,created_at,test_week,clickability_test_id,headline,eyecatcher_id,impressions,clicks,ctr,ctr_demeaned,...,neg,neu,pos,compound,created_at_dayofweek,created_at_hourofday,test_group_size,read_flesch,read_coleman,specificity_tfidf
0,62,2014-11-20 14:57:52.478,201446,546e009a9ad54ec65b00004b,What They Learned From The Scientist Was Terri...,546c7f2dbadeb5788700000a,4594,51,0.011101,0.111516,...,0.144,0.856,0.000,-0.3291,3,14,6,80.782500,8.093333,0.444616
1,84,2014-11-20 14:54:18.780,201446,546e009a9ad54ec65b00004b,A Science Guy Helps 3 Dudes From America Under...,546c7f2dbadeb5788700000a,4571,58,0.012689,0.682108,...,0.000,0.809,0.191,0.3818,3,14,6,59.682143,10.257143,0.370851
2,95,2014-11-20 15:04:49.517,201446,546e009a9ad54ec65b00004b,He Sat Them Down And Told Them About An Immine...,546c7f2dbadeb5788700000a,4601,27,0.005868,-1.769714,...,0.187,0.813,0.000,-0.5994,3,15,6,80.465000,6.077778,0.398801
3,100,2014-11-20 15:13:36.266,201446,546e009a9ad54ec65b00004b,"The 3 Of Them Needed To See It In Person, And ...",546c7f2dbadeb5788700000a,4567,63,0.013795,1.079669,...,0.124,0.720,0.156,0.2023,3,15,6,80.777143,3.228571,0.445514
4,102,2014-11-20 15:15:25.697,201446,546e009a9ad54ec65b00004b,"They May Not Be The Most Handsome Dudes, But T...",546c7f2dbadeb5788700000a,4524,44,0.009726,-0.382964,...,0.000,0.655,0.345,0.7812,3,15,6,92.965000,5.875000,0.497006


## BERTopic

I decided to use the `all-MiniLM-L6-v2` model for the embeddings as it's a simple model for this task with a good performance track and recommended by the library documentation.

In [7]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

exploratory_headlines = exploratory_df["headline"].fillna("").tolist()
confirmatory_headlines = confirmatory_df["headline"].fillna("").tolist()
holdout_headlines = holdout_df["headline"].fillna("").tolist()

# Create the SentenceTransformer embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Fit the BERTTopic model
# Set min_topic_size so only topics with at least 80 headlines are considered
# For IF-IDF I use unigrams and bigrams
topic_model = BERTopic(
    embedding_model=embedding_model,
    n_gram_range=(1, 2),
    min_topic_size=40,
    top_n_words=20,
    verbose=True,
)

exploratory_topics, exploratory_probs = topic_model.fit_transform(exploratory_headlines)

exploratory_df["topic_bertopic"] = exploratory_topics

# Use the fitted model to assign topics to confirmatory and holdout sets
confirmatory_headlines = confirmatory_df["headline"].fillna("").tolist()
holdout_headlines = holdout_df["headline"].fillna("").tolist()
confirmatory_topics, confirmatory_probs = topic_model.transform(confirmatory_headlines)
holdout_topics, holdout_probs = topic_model.transform(holdout_headlines)
confirmatory_df["topic_bertopic"] = confirmatory_topics
holdout_df["topic_bertopic"] = holdout_topics

exploratory_df[["headline", "topic_bertopic"]].head()


2025-12-14 17:23:29,329 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/376 [00:00<?, ?it/s]

2025-12-14 17:23:42,531 - BERTopic - Embedding - Completed ✓
2025-12-14 17:23:42,533 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-12-14 17:23:45,179 - BERTopic - Dimensionality - Completed ✓
2025-12-14 17:23:45,182 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-12-14 17:23:45,575 - BERTopic - Cluster - Completed ✓
2025-12-14 17:23:45,587 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-12-14 17:23:45,897 - BERTopic - Representation - Completed ✓


Batches:   0%|          | 0/1795 [00:00<?, ?it/s]

2025-12-14 17:24:45,486 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2025-12-14 17:24:55,176 - BERTopic - Dimensionality - Completed ✓
2025-12-14 17:24:55,178 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2025-12-14 17:24:57,087 - BERTopic - Cluster - Completed ✓


Batches:   0%|          | 0/390 [00:00<?, ?it/s]

2025-12-14 17:25:11,253 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2025-12-14 17:25:13,331 - BERTopic - Dimensionality - Completed ✓
2025-12-14 17:25:13,332 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2025-12-14 17:25:13,797 - BERTopic - Cluster - Completed ✓


,headline,topic_bertopic
0,What They Learned From The Scientist Was Terri...,4
1,A Science Guy Helps 3 Dudes From America Under...,4
2,He Sat Them Down And Told Them About An Immine...,-1
3,"The 3 Of Them Needed To See It In Person, And ...",-1
4,"They May Not Be The Most Handsome Dudes, But T...",-1


In [8]:
# Print the number of headlines per topic
print(f"Exploratory set: Number of headlines per topic ({len(exploratory_df)} headlines):")
print(exploratory_df["topic_bertopic"].value_counts().head(20))

print(f"Confirmatory set: Number of headlines per topic ({len(confirmatory_df)} headlines):")
print(confirmatory_df["topic_bertopic"].value_counts().head(20))

print(f"Holdout set: Number of headlines per topic ({len(holdout_df)} headlines):")
print(holdout_df["topic_bertopic"].value_counts().head(20))


Exploratory set: Number of headlines per topic (12010 headlines):
topic_bertopic
-1     5865
 0      854
 1      402
 2      318
 3      311
 4      217
 5      210
 6      191
 7      190
 8      186
 9      185
 10     149
 11     147
 12     145
 13     133
 14     121
 15     121
 16     118
 17     110
 18     105
Name: count, dtype: int64
Confirmatory set: Number of headlines per topic (57413 headlines):
topic_bertopic
-1     28878
 0      3864
 1      1931
 2      1910
 3      1514
 5      1237
 4      1101
 9       916
 6       867
 7       831
 8       814
 16      683
 12      661
 11      611
 10      563
 26      543
 14      542
 21      538
 17      519
 22      501
Name: count, dtype: int64
Holdout set: Number of headlines per topic (12474 headlines):
topic_bertopic
-1     6043
 0      794
 2      415
 1      415
 3      381
 5      319
 9      203
 4      201
 12     199
 21     185
 6      181
 8      171
 10     152
 11     150
 7      144
 24     135
 16     134
 17 

Around half the headlines don't have a topic assigned. This is ok, we only care about the topics we can really identify rather than noise.

The good thing is that the topics discovered using the exploratory set were fitted to the confirmatory and holdout sets in similar proportions.

### Topics Analysis

In [9]:
# Inspect the discovered topics

# Overview of the 20 most common topics
topic_info = topic_model.get_topic_info()
topic_info.head(20)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,5865,-1_the_to_you_of,"[the, to, you, of, it, this, and, in, is, that...",[If You Don't Like The Way Things Are In This ...
1,0,854,0_she_her_to_the,"[she, her, to, the, mom, was, and, his, it, pa...",[OMG. They Tell Her That Her Words Are BS Righ...
2,1,402,1_women_men_feminist_to,"[women, men, feminist, to, the, these, of, and...",[‘All Men And Women Are Created Equal’ … Psych...
3,2,318,2_gay_straight_marriage_gay marriage,"[gay, straight, marriage, gay marriage, the, r...",[A Pastor Asks His Politician Why He Supports ...
4,3,311,3_food_eat_fast food_mcdonald,"[food, eat, fast food, mcdonald, you, fast, re...",[Fast Food Usually Grosses Me Out. But This Vi...
5,4,217,4_science_doctor_the_facts,"[science, doctor, the, facts, scientist, all, ...","[16 Years Ago, A Doctor Published A Study. It ..."
6,5,210,5_black_white_race_racist,"[black, white, race, racist, racism, people, b...",[If You're Wondering How To Talk About Black P...
7,6,191,6_kids_these kids_kid_these,"[kids, these kids, kid, these, kids are, to, w...",[These kids aren't all right. 7 beautiful phot...
8,7,190,7_fashion_her_model_beauty,"[fashion, her, model, beauty, she, jennifer la...","[Nope, Jennifer Lawrence Isn't 'Sorry' For Her..."
9,8,186,8_song_music_this song_rap,"[song, music, this song, rap, of, rapper, it, ...",[One Of The Best Parts Of This Song Is That It...


Looking at the most frequent words in the most common topics we easily interpret what most of them are about:

- –1: Miscellaneous
- 0: She, He, Pronouns
- 1: Kids
- 2: Feminism
- 3: Gay Marriage
- 4: Viral Videos & Content
- 5: Race & White People
- 6: Food & Restaurants
- 7: Fox News
- 8: Jobs & Money
- 9: Music
- 10: Water & City
- 11: Science
- 12: Rape & Sexual Violence
- 13: Abortion & Reproductive Rights
- 14: Football & Sports
- 15: Climate Change
- 16: Space
- 17: Fashion
- 18: Minimum Wage

Now I'll save the dataset

In [10]:
exploratory_df.to_csv("dataset/processed/exploratory-packages-topic-modeling.csv", index=False)
confirmatory_df.to_csv("dataset/processed/confirmatory-packages-topic-modeling.csv", index=False)
holdout_df.to_csv("dataset/processed/holdout-packages-topic-modeling.csv", index=False)

For future reference, I'll save all the topics with the most frequent words.

In [11]:
# Save the topics
topics = topic_model.get_topics()

rows = []
for topic_id, word_weights in topics.items():
    # Take the top 50 words for this topic
    top_words = [word for word, _ in word_weights[:50]]
    rows.append({
        "topic_id": topic_id,
        "top_50_words": ", ".join(top_words),
    })

topics_df = pd.DataFrame(rows).sort_values("topic_id")

output_path = "dataset/processed/topics.csv"
topics_df.to_csv(output_path, index=False)
topics_df.head()

,topic_id,top_50_words
0,-1,"the, to, you, of, it, this, and, in, is, that,..."
1,0,"she, her, to, the, mom, was, and, his, it, par..."
2,1,"women, men, feminist, to, the, these, of, and,..."
3,2,"gay, straight, marriage, gay marriage, the, re..."
4,3,"food, eat, fast food, mcdonald, you, fast, res..."
